In [0]:
# --------------------------------------------------------------
# Set environment variables for the pipeline run
# --------------------------------------------------------------
# These values are retrieved from the master pipeline task 
# (superstore_pipeline_master_run_id_init) using dbutils.jobs.taskValues.
# They ensure that the ETL, Mart, Features, KPI, and Dashboard code
# runs in the correct environment (dev/qa/prod) without hardcoding paths.
# SUPERSTORE_ENV, SUPERSTORE_PIPELINE_NAME, and SUPERSTORE_PIPELINE_VERSION
# are set for downstream modules to dynamically resolve catalog/schema.
# --------------------------------------------------------------
import os

env = dbutils.jobs.taskValues.get(taskKey="superstore_pipeline_master_run_id_init", key="SUPERSTORE_ENV")
pipeline_name = dbutils.jobs.taskValues.get(taskKey="superstore_pipeline_master_run_id_init", key="SUPERSTORE_PIPELINE_NAME")
pipeline_version = dbutils.jobs.taskValues.get(taskKey="superstore_pipeline_master_run_id_init", key="SUPERSTORE_PIPELINE_VERSION")

os.environ["SUPERSTORE_ENV"] = env
os.environ["SUPERSTORE_PIPELINE_NAME"] = pipeline_name
os.environ["SUPERSTORE_PIPELINE_VERSION"] = pipeline_version

# -------------------------------
# Environment Config
# -------------------------------
import sys
sys.path.append("/Workspace/Users/bireshmoktan@gmail.com/superstore_medallionarchitecture_dab/src/superstore_shared_utilities")
from superstore_platform_config import get_catalog, get_gold_schema, get_mart_schema, get_features_schema

# Dynamically get environment-aware catalog and schemas
catalog = get_catalog()            # e.g., superstore_catalog
gold_schema = get_gold_schema()    # e.g., dev_gold
features_schema = get_features_schema() 

# -------------------------------
# Create feature customer Table (Environment-aware)
# -------------------------------
spark.sql(f"""
-- Create Customer ML Features Table
CREATE OR REPLACE TABLE {catalog}.{features_schema}.features_customer AS

WITH customer_orders AS (
    SELECT
        c.customer_id,
        o.order_id,
        o.order_date,
        sum(s.sales) as total_sales,
        sum(s.profit) as total_profit,
        avg(s.discount) as avg_discount,
        c.region,
        c.segment
    FROM {catalog}.{gold_schema}.dim_customers c 
    join {catalog}.{gold_schema}.facts_orders o on c.customer_id= o.customer_id
    join {catalog}.{gold_schema}.facts_sales s on o.order_id= s.order_id
    group by c.customer_id, o.order_id, o.order_date,c.region, c.segment
),

-- 1️.Recency: Days since last order per customer
customer_recency AS (
    SELECT
        customer_id,
        CURRENT_DATE() AS feature_date,
        DATEDIFF(CURRENT_DATE(), MAX(order_date)) AS recency_days
    FROM customer_orders
    GROUP BY customer_id
),

-- 2️.Order frequency in last 30 & 90 days
customer_frequency AS (
    SELECT
        customer_id,
        SUM(CASE WHEN order_date >= DATE_SUB(CURRENT_DATE(), 30) THEN 1 ELSE 0 END) AS order_frequency_30days,
        SUM(CASE WHEN order_date >= DATE_SUB(CURRENT_DATE(), 90) THEN 1 ELSE 0 END) AS order_frequency_90days
    FROM customer_orders
    GROUP BY customer_id
),

-- 3️.Monetary value in last 30 & 90 days
customer_monetary AS (
    SELECT
        customer_id,
        SUM(CASE WHEN order_date >= DATE_SUB(CURRENT_DATE(), 30) THEN total_sales ELSE 0 END) AS monetary_30days,
        SUM(CASE WHEN order_date >= DATE_SUB(CURRENT_DATE(), 90) THEN total_sales ELSE 0 END) AS monetary_90days,
        AVG(avg_discount) AS avg_discount
    FROM customer_orders
    GROUP BY customer_id
),

-- 4️.Profit margin & order variance
customer_profit_variance AS (
    SELECT
        customer_id,
        CASE 
            WHEN SUM(total_sales) = 0 THEN 0
            ELSE ROUND(SUM(total_profit)/SUM(total_sales),4)
        END AS profit_margin,
        CASE 
            WHEN COUNT(order_id) > 1 THEN ROUND(VAR_SAMP(total_sales),2)
            ELSE 0
        END AS order_variance
    FROM customer_orders
    GROUP BY customer_id
),

-- 5️.Encode segment & region (simple encoding: map to integers)
customer_encoding AS (
    SELECT
        customer_id,
        DENSE_RANK() OVER (ORDER BY segment) AS segment_encoded,
        DENSE_RANK() OVER (ORDER BY region) AS region_encoded
    FROM customer_orders
    GROUP BY customer_id, segment, region
)

-- 6️.Combine all features into final table
SELECT
    r.customer_id,
    r.feature_date,
    r.recency_days,
    f.order_frequency_30days,
    f.order_frequency_90days,
    m.monetary_30days,
    m.monetary_90days,
    m.avg_discount,
    p.profit_margin,
    p.order_variance,
    e.segment_encoded,
    e.region_encoded
FROM customer_recency r
JOIN customer_frequency f ON r.customer_id = f.customer_id
JOIN customer_monetary m ON r.customer_id = m.customer_id
JOIN customer_profit_variance p ON r.customer_id = p.customer_id
JOIN customer_encoding e ON r.customer_id = e.customer_id;
""")
